In [1]:
import torch
import torch.nn as nn # 다양한 신경망 함수, 활성함수, 손실함수등을 불러올수있는 라이브러리이다.
import torch.optim as optim

# 시드 고정
torch.manual_seed(0)

# ----- 1. 데이터 준비 -----
chars = ["H", "E", "L", "O"]
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

# HELLO -> 다음 글자 예측
X_data = [char_to_idx[c] for c in "HELL"]  # 입력: H, E, L, L
Y_data = [char_to_idx[c] for c in "ELLO"]  # 정답: E, L, L, O

# 원핫 벡터로 만들기
def one_hot(x):
    oh = torch.zeros(len(chars))
    oh[x] = 1.0
    return oh

X = torch.stack([one_hot(x) for x in X_data])
Y = torch.tensor(Y_data)

# ----- 2. 모델 정의 -----
input_size = len(chars)   # 4
hidden_size = 8
output_size = len(chars)  # 4

rnn = nn.RNNCell(input_size, hidden_size) # tanh활성함수가 자동으로 포함되어있다.
fc = nn.Linear(hidden_size, output_size)

criterion = nn.CrossEntropyLoss() # softmax가 내포되어있음.
optimizer = optim.Adam(list(rnn.parameters()) + list(fc.parameters()), lr=0.05)

# ----- 3. 학습 -----
print("=== Training Start ===\n")
for epoch in range(50):
    hidden = torch.zeros(hidden_size)

    total_loss = 0
    outputs = []

    print(f"\n--- Epoch {epoch+1} ---")

    for t in range(len(X)):
        # RNNCell 한 스텝
        hidden = rnn(X[t], hidden)

        # 출력층
        out = fc(hidden)
        outputs.append(out)

        # 출력값을 문자로 변환해서 보여주기
        pred_idx = torch.argmax(out).item()
        print(f"Step {t}: Input={idx_to_char[X_data[t]]}, "
              f"Target={idx_to_char[Y_data[t]]}, "
              f"Pred={idx_to_char[pred_idx]}")

        # 손실 계산
        loss = criterion(out.unsqueeze(0), Y[t].unsqueeze(0))
        total_loss += loss

    # 역전파
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    print(f"Epoch Loss: {total_loss.item():.4f}")

print("\n=== Training End ===")

# ----- 4. 최종 예측 확인 -----
print("\n=== Final Predictions ===")
hidden = torch.zeros(hidden_size)
for t in range(len(X)):
    hidden = rnn(X[t], hidden)
    out = fc(hidden)
    pred_idx = torch.argmax(out).item()
    print(f"Input={idx_to_char[X_data[t]]} → Predicted={idx_to_char[pred_idx]}")


=== Training Start ===


--- Epoch 1 ---
Step 0: Input=H, Target=E, Pred=L
Step 1: Input=E, Target=L, Pred=O
Step 2: Input=L, Target=L, Pred=L
Step 3: Input=L, Target=O, Pred=L
Epoch Loss: 5.3765

--- Epoch 2 ---
Step 0: Input=H, Target=E, Pred=L
Step 1: Input=E, Target=L, Pred=L
Step 2: Input=L, Target=L, Pred=L
Step 3: Input=L, Target=O, Pred=L
Epoch Loss: 4.7004

--- Epoch 3 ---
Step 0: Input=H, Target=E, Pred=L
Step 1: Input=E, Target=L, Pred=L
Step 2: Input=L, Target=L, Pred=L
Step 3: Input=L, Target=O, Pred=L
Epoch Loss: 4.1696

--- Epoch 4 ---
Step 0: Input=H, Target=E, Pred=L
Step 1: Input=E, Target=L, Pred=L
Step 2: Input=L, Target=L, Pred=L
Step 3: Input=L, Target=O, Pred=L
Epoch Loss: 3.7010

--- Epoch 5 ---
Step 0: Input=H, Target=E, Pred=L
Step 1: Input=E, Target=L, Pred=L
Step 2: Input=L, Target=L, Pred=L
Step 3: Input=L, Target=O, Pred=L
Epoch Loss: 3.2656

--- Epoch 6 ---
Step 0: Input=H, Target=E, Pred=E
Step 1: Input=E, Target=L, Pred=L
Step 2: Input=L, Target=L, Pred

In [2]:

from tqdm import tqdm
import numpy as np
import re
##### Data #####
data = "나라의 말이 중국과 달라 문자와 서로 통하지 아니하기에 이런 까닭으로 어리석은 백성이 이르고자 할 바가 있어도 마침내 제 뜻을 능히 펴지 못할 사람이 많으니라 내가 이를 위해 가엾이 여겨 새로 스물여덟 글자를 만드노니 사람마다 하여 쉬이 익혀 날로 씀에 편안케 하고자 할 따름이니라"
#데이터를 preprocessing 해주는 부분입니다
#RNN과 동일한 방법입니다
def data_preprocessing(data):
    data = re.sub('[^가-힣]', ' ', data)
    tokens = data.split()
    vocab = list(set(tokens))
    vocab_size = len(vocab)

    Word_to_ix = {Word: i for i, Word in enumerate(vocab)}
    ix_to_Word = {i: Word for i, Word in enumerate(vocab)}

    return tokens, vocab_size, Word_to_ix, ix_to_Word
# 활성화 함수
def sigmoid(input):
    return 1 / (1 + np.exp(-input))

def sigmoid_derivative(input):
    return input * (1 - input)
    
def tanh(input, derivative=False):
    return np.tanh(input)

def tanh_derivative(input):
    return 1 - input ** 2

def softmax(input):
    return np.exp(input) / np.sum(np.exp(input))
class LSTM:
    def __init__(self, input_size, hidden_size, output_size, num_epochs, learning_rate):
        # Hyperparameters
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.num_epochs = num_epochs

        # Forget Gate
        self.Wf = np.random.randn(hidden_size, input_size)*0.1
        self.bf = np.zeros((hidden_size, 1))

        # Input Gate
        self.Wi = np.random.randn(hidden_size, input_size)*0.1
        self.bi = np.zeros((hidden_size, 1))

        # Candidate Gate
        self.Wc = np.random.randn(hidden_size, input_size)*0.1
        self.bc = np.zeros((hidden_size, 1))

        # Output Gate
        self.Wo = np.random.randn(hidden_size, input_size)*0.1
        self.bo = np.zeros((hidden_size, 1))

        # Final Gate
        self.Wy = np.random.randn(output_size, hidden_size)
        self.by = np.zeros((output_size, 1))
        
    # 네트워크 메모리 리셋
    def reset(self):
        self.X = {}

        self.HS = {-1: np.zeros((self.hidden_size, 1))}
        self.CS = {-1: np.zeros((self.hidden_size, 1))}

        self.C = {}
        self.O = {}
        self.F = {}
        self.I = {}
        self.outputs = {}

    # Forward 순전파
    def forward(self, inputs):
        # self.reset()
        x = {}
        outputs = []
        for t in range(len(inputs)):
            x[t] = np.zeros((vocab_size , 1))
            x[t][inputs[t]] = 1  # 각각의 Word에 대한 one hot coding
            self.X[t] = np.concatenate((self.HS[t - 1], x[t]))

            self.F[t] = sigmoid(np.dot(self.Wf, self.X[t]) + self.bf)
            self.I[t] = sigmoid(np.dot(self.Wi, self.X[t]) + self.bi)
            self.C[t] = tanh(np.dot(self.Wc, self.X[t]) + self.bc)
            self.O[t] = sigmoid(np.dot(self.Wo, self.X[t]) + self.bo)

            self.CS[t] = self.F[t] * self.CS[t - 1] + self.I[t] * self.C[t]
            self.HS[t] = self.O[t] * tanh(self.CS[t])

            outputs += [np.dot(self.Wy, self.HS[t]) + self.by]

        return outputs

    # 역전파
    def backward(self, errors, inputs):
        dLdWf, dLdbf = 0, 0
        dLdWi, dLdbi = 0, 0
        dLdWc, dLdbc = 0, 0
        dLdWo, dLdbo = 0, 0
        dLdWy, dLdby = 0, 0

        dh_next, dc_next = np.zeros_like(self.HS[0]), np.zeros_like(self.CS[0])
        for t in reversed(range(len(inputs))):
            error = errors[t]

            # Final Gate Weights and Biases Errors
            dLdWy += np.dot(error, self.HS[t].T)         #𝜕𝐿/𝜕𝑊𝑦
            dLdby += error                               #𝜕𝐿/𝜕b𝑦 = (𝜕𝐿/𝜕z_t)(𝜕z_t/𝜕b𝑦) = error x 1 (Zt = WyHSt + by)
            
            # Hidden State Error
            dLdHS = np.dot(self.Wy.T, error) + dh_next    #𝜕𝐿/𝜕𝐻𝑆

            # Output Gate Weights and Biases Errors
            dLdo = tanh(self.CS[t]) * dLdHS * sigmoid_derivative(self.O[t])
            dLdWo += np.dot(dLdo, inputs[t].T)
            dLdbo += dLdo

            # Cell State Error
            dLdCS = tanh_derivative(tanh(self.CS[t])) * self.O[t] * dLdHS + dc_next

            # Forget Gate Weights and Biases Errors
            dLdf = dLdCS * self.CS[t - 1] * sigmoid_derivative(self.F[t])
            dLdWf += np.dot(dLdf, inputs[t].T)
            dLdbf += dLdf

            # Input Gate Weights and Biases Errors
            dLdi = dLdCS * self.C[t] * sigmoid_derivative(self.I[t])
            dLdWi += np.dot(dLdi, inputs[t].T)
            dLdbi += dLdi

            # Candidate Gate Weights and Biases Errors
            dLdc = dLdCS * self.I[t] * tanh_derivative(self.C[t])
            dLdWc += np.dot(dLdc, inputs[t].T)
            dLdbc += dLdc

            # Concatenated Input Error (Sum of Error at Each Gate!)
            d_z = np.dot(self.Wf.T, dLdf) + np.dot(self.Wi.T, dLdi) + np.dot(self.Wc.T, dLdc) + np.dot(self.Wo.T, dLdo)

            # Error of Hidden State and Cell State at Next Time Step
            dh_next = d_z[:self.hidden_size, :]
            dc_next = self.F[t] * dLdCS
            
        for d_ in (dLdWf, dLdbf, dLdWi, dLdbi, dLdWc, dLdbc, dLdWo, dLdbo, dLdWy, dLdby):
            np.clip(d_, -1, 1, out=d_)

        self.Wf += dLdWf * self.learning_rate * (-1)
        self.bf += dLdbf * self.learning_rate * (-1)

        self.Wi += dLdWi * self.learning_rate * (-1)
        self.bi += dLdbi * self.learning_rate * (-1)

        self.Wc += dLdWc * self.learning_rate * (-1)
        self.bc += dLdbc * self.learning_rate * (-1)

        self.Wo += dLdWo * self.learning_rate * (-1)
        self.bo += dLdbo * self.learning_rate * (-1)

        self.Wy += dLdWy * self.learning_rate * (-1)
        self.by += dLdby * self.learning_rate * (-1)

    # Train
    def train(self, inputs, labels):
        for _ in tqdm(range(self.num_epochs)):
            self.reset()
            input_idx = [Word_to_ix[input] for input in inputs]
            predictions = self.forward(input_idx)

            errors = []
            for t in range(len(predictions)):
                errors += [softmax(predictions[t])]
                errors[-1][Word_to_ix[labels[t]]] -= 1

            self.backward(errors, self.X)

    def test(self, inputs, labels):
        accuracy = 0
        probabilities = self.forward([Word_to_ix[input] for input in inputs])

        gt = ''
        output = '나라의 '
        for q in range(len(labels)):
            prediction = ix_to_Word[np.argmax(softmax(probabilities[q].reshape(-1)))]
            gt += inputs[q] + ' '
            output += prediction + ' '
            
            if prediction == labels[q]:
                accuracy += 1

        print('실제값: ', gt)
        print('예측값: ', output)
hidden_size = 25

# data preparation
tokens, vocab_size, Word_to_ix, ix_to_Word = data_preprocessing(data)
train_X, train_y = tokens[:-1], tokens[1:]

lstm = LSTM(input_size=vocab_size + hidden_size, hidden_size=hidden_size, output_size=vocab_size, num_epochs=1000,
            learning_rate=0.05)

##### Training #####
lstm.train(train_X, train_y)

lstm.test(train_X, train_y)


100%|██████████████████████████████████████| 1000/1000 [00:02<00:00, 493.23it/s]

실제값:  나라의 말이 중국과 달라 문자와 서로 통하지 아니하기에 이런 까닭으로 어리석은 백성이 이르고자 할 바가 있어도 마침내 제 뜻을 능히 펴지 못할 사람이 많으니라 내가 이를 위해 가엾이 여겨 새로 스물여덟 글자를 만드노니 사람마다 하여 쉬이 익혀 날로 씀에 편안케 하고자 할 
예측값:  나라의 말이 중국과 달라 문자와 서로 통하지 아니하기에 이런 까닭으로 어리석은 백성이 이르고자 할 바가 있어도 마침내 제 뜻을 능히 펴지 못할 사람이 많으니라 내가 이를 위해 가엾이 여겨 새로 스물여덟 글자를 만드노니 사람마다 하여 쉬이 익혀 날로 씀에 편안케 하고자 할 따름이니라 


In [10]:
from tqdm import tqdm
import numpy as np
import re

##### Data #####
data = "나라의 말이 중국과 달라 문자와 서로 통하지 아니하기에 이런 까닭으로 어리석은 백성이 이르고자 할 바가 있어도 마침내 제 뜻을 능히 펴지 못할 사람이 많으니라 내가 이를 위해 가엾이 여겨 새로 스물여덟 글자를 만드노니 사람마다 하여 쉬이 익혀 날로 씀에 편안케 하고자 할 따름이니라"

# 데이터 전처리 (RNN과 동일)
def data_preprocessing(data):
    data = re.sub('[^가-힣]', ' ', data)
    tokens = data.split()
    vocab = list(set(tokens))
    vocab_size = len(vocab)

    Word_to_ix = {Word: i for i, Word in enumerate(vocab)}
    ix_to_Word = {i: Word for i, Word in enumerate(vocab)}

    return tokens, vocab_size, Word_to_ix, ix_to_Word

# 활성화 함수들
def sigmoid(input):
    return 1 / (1 + np.exp(-input))

def sigmoid_derivative(input):
    return input * (1 - input)
    
def tanh(input, derivative=False):
    return np.tanh(input)

def tanh_derivative(input):
    return 1 - input ** 2

def softmax(input):
    # 안정성을 위해 최대값 빼주기
    input = input - np.max(input)
    exps = np.exp(input)
    return exps / np.sum(exps)

# 🔹 문장 전체에 대한 cross-entropy loss 계산 함수
def compute_loss(predictions, labels):
    loss = 0.0
    for t in range(len(predictions)):
        probs = softmax(predictions[t]).reshape(-1)  # (vocab_size,) 로 펴주기
        y_idx = Word_to_ix[labels[t]]
        loss -= np.log(probs[y_idx] + 1e-9)         # cross-entropy
    return float(loss)  # 확실히 스칼라로 캐스팅

class LSTM:
    def __init__(self, input_size, hidden_size, output_size, num_epochs, learning_rate):
        # Hyperparameters
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.num_epochs = num_epochs

        # Forget Gate
        self.Wf = np.random.randn(hidden_size, input_size)*0.1
        self.bf = np.zeros((hidden_size, 1))

        # Input Gate
        self.Wi = np.random.randn(hidden_size, input_size)*0.1
        self.bi = np.zeros((hidden_size, 1))

        # Candidate Gate
        self.Wc = np.random.randn(hidden_size, input_size)*0.1
        self.bc = np.zeros((hidden_size, 1))

        # Output Gate
        self.Wo = np.random.randn(hidden_size, input_size)*0.1
        self.bo = np.zeros((hidden_size, 1))

        # Final Gate (output layer)
        self.Wy = np.random.randn(output_size, hidden_size)*0.1
        self.by = np.zeros((output_size, 1))
        
    # 네트워크 메모리 리셋
    def reset(self):
        self.X = {}

        self.HS = {-1: np.zeros((self.hidden_size, 1))}
        self.CS = {-1: np.zeros((self.hidden_size, 1))}

        self.C = {}
        self.O = {}
        self.F = {}
        self.I = {}
        self.outputs = {}

    # Forward 순전파
    def forward(self, inputs):
        x = {}
        outputs = []
        for t in range(len(inputs)):
            x[t] = np.zeros((vocab_size , 1))
            x[t][inputs[t]] = 1  # 각각의 Word에 대한 one-hot encoding
            self.X[t] = np.concatenate((self.HS[t - 1], x[t]))

            self.F[t] = sigmoid(np.dot(self.Wf, self.X[t]) + self.bf)
            self.I[t] = sigmoid(np.dot(self.Wi, self.X[t]) + self.bi)
            self.C[t] = tanh(np.dot(self.Wc, self.X[t]) + self.bc)
            self.O[t] = sigmoid(np.dot(self.Wo, self.X[t]) + self.bo)

            self.CS[t] = self.F[t] * self.CS[t - 1] + self.I[t] * self.C[t]
            self.HS[t] = self.O[t] * tanh(self.CS[t])

            outputs += [np.dot(self.Wy, self.HS[t]) + self.by]

        return outputs

    # 역전파
    def backward(self, errors, inputs):
        dLdWf, dLdbf = 0, 0
        dLdWi, dLdbi = 0, 0
        dLdWc, dLdbc = 0, 0
        dLdWo, dLdbo = 0, 0
        dLdWy, dLdby = 0, 0

        dh_next, dc_next = np.zeros_like(self.HS[0]), np.zeros_like(self.CS[0])
        for t in reversed(range(len(inputs))):
            error = errors[t]  # (vocab_size, 1)

            # Final Gate Weights and Biases Errors
            dLdWy += np.dot(error, self.HS[t].T)         # ∂L/∂Wy
            dLdby += error                               # ∂L/∂by
            
            # Hidden State Error
            dLdHS = np.dot(self.Wy.T, error) + dh_next   # ∂L/∂HS

            # Output Gate Weights and Biases Errors
            dLdo = tanh(self.CS[t]) * dLdHS * sigmoid_derivative(self.O[t])
            dLdWo += np.dot(dLdo, inputs[t].T)
            dLdbo += dLdo

            # Cell State Error
            dLdCS = tanh_derivative(tanh(self.CS[t])) * self.O[t] * dLdHS + dc_next

            # Forget Gate Weights and Biases Errors
            dLdf = dLdCS * self.CS[t - 1] * sigmoid_derivative(self.F[t])
            dLdWf += np.dot(dLdf, inputs[t].T)
            dLdbf += dLdf

            # Input Gate Weights and Biases Errors
            dLdi = dLdCS * self.C[t] * sigmoid_derivative(self.I[t])
            dLdWi += np.dot(dLdi, inputs[t].T)
            dLdbi += dLdi

            # Candidate Gate Weights and Biases Errors
            dLdc = dLdCS * self.I[t] * tanh_derivative(self.C[t])
            dLdWc += np.dot(dLdc, inputs[t].T)
            dLdbc += dLdc

            # Concatenated Input Error (Sum of Error at Each Gate!)
            d_z = (np.dot(self.Wf.T, dLdf) + 
                   np.dot(self.Wi.T, dLdi) + 
                   np.dot(self.Wc.T, dLdc) + 
                   np.dot(self.Wo.T, dLdo))

            # Error of Hidden State and Cell State at Next Time Step
            dh_next = d_z[:self.hidden_size, :]
            dc_next = self.F[t] * dLdCS
            
        # Gradient clipping
        for d_ in (dLdWf, dLdbf, dLdWi, dLdbi, dLdWc, dLdbc, dLdWo, dLdbo, dLdWy, dLdby):
            np.clip(d_, -1, 1, out=d_)

        # Gradient descent
        self.Wf -= self.learning_rate * dLdWf
        self.bf -= self.learning_rate * dLdbf

        self.Wi -= self.learning_rate * dLdWi
        self.bi -= self.learning_rate * dLdbi

        self.Wc -= self.learning_rate * dLdWc
        self.bc -= self.learning_rate * dLdbc

        self.Wo -= self.learning_rate * dLdWo
        self.bo -= self.learning_rate * dLdbo

        self.Wy -= self.learning_rate * dLdWy
        self.by -= self.learning_rate * dLdby

    # Train
    def train(self, inputs, labels, print_every=100):
        for epoch in tqdm(range(1, self.num_epochs + 1)):
            self.reset()
            input_idx = [Word_to_ix[input] for input in inputs]

            # 1) 순전파
            predictions = self.forward(input_idx)

            # 2) 문장 전체에 대한 loss 계산
            loss = compute_loss(predictions, labels)

            # 3) Softmax + Cross-Entropy gradient 생성
            errors = []
            for t in range(len(predictions)):
                probs = softmax(predictions[t]).reshape(-1, 1)  # (vocab_size, 1)
                probs[Word_to_ix[labels[t]]] -= 1               # y_hat - y
                errors.append(probs)

            # 4) 역전파
            self.backward(errors, self.X)

            # 5) 에포크마다 loss 출력
            if epoch % print_every == 0 or epoch == 1:
                print(f"Epoch {epoch}/{self.num_epochs} - Loss: {loss:.4f}")

    def test(self, inputs, labels):
        accuracy = 0
        probabilities = self.forward([Word_to_ix[input] for input in inputs])

        gt = ''
        output = '나라의 '
        for q in range(len(labels)):
            prediction = ix_to_Word[np.argmax(softmax(probabilities[q].reshape(-1)))]
            gt += inputs[q] + ' '
            output += prediction + ' '
            
            if prediction == labels[q]:
                accuracy += 1

        print('실제값: ', gt)
        print('예측값: ', output)
        print(f'정확도: {accuracy}/{len(labels)} = {accuracy/len(labels):.2f}')


##### Hyperparameters #####
hidden_size = 25

# data preparation
tokens, vocab_size, Word_to_ix, ix_to_Word = data_preprocessing(data)
train_X, train_y = tokens[:-1], tokens[1:]

lstm = LSTM(
    input_size=vocab_size + hidden_size,
    hidden_size=hidden_size,
    output_size=vocab_size,
    num_epochs=200,
    learning_rate=0.05
)

##### Training #####
lstm.train(train_X, train_y, print_every=100)

##### Test #####
lstm.test(train_X, train_y)


 38%|███████████████▌                         | 76/200 [00:00<00:00, 377.92it/s]

Epoch 1/200 - Loss: 156.7486


 80%|████████████████████████████████        | 160/200 [00:00<00:00, 404.14it/s]

Epoch 100/200 - Loss: 41.2755


100%|████████████████████████████████████████| 200/200 [00:00<00:00, 400.23it/s]

Epoch 200/200 - Loss: 5.4153
실제값:  나라의 말이 중국과 달라 문자와 서로 통하지 아니하기에 이런 까닭으로 어리석은 백성이 이르고자 할 바가 있어도 마침내 제 뜻을 능히 펴지 못할 사람이 많으니라 내가 이를 위해 가엾이 여겨 새로 스물여덟 글자를 만드노니 사람마다 하여 쉬이 익혀 날로 씀에 편안케 하고자 할 
예측값:  나라의 말이 중국과 달라 문자와 서로 통하지 아니하기에 이런 까닭으로 어리석은 백성이 이르고자 할 바가 있어도 마침내 제 뜻을 능히 펴지 못할 사람이 많으니라 내가 이를 위해 가엾이 여겨 새로 스물여덟 글자를 만드노니 사람마다 하여 쉬이 익혀 날로 씀에 편안케 하고자 할 따름이니라 
정확도: 42/42 = 1.00


# word 2 vec

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# 재현 가능성을 위해 시드 고정 (선택)
torch.manual_seed(0)
random.seed(0)

# -----------------------------
# 1. 예시용 vocab / 데이터 준비
# -----------------------------
V = 10  # vocab size

pairs = [
    (0, 1),
    (0, 2),
    (1, 3),
    (2, 4),
    (3, 5),
    (4, 6),
    (5, 7),
    (6, 8),
    (7, 9),
    (8, 0),
]

# -----------------------------
# 2. Negative Sampling 함수
# -----------------------------
def get_negative_samples(pos_idx, K):
    """
    pos_idx: 현재 positive context index (제외해야 하는 단어)
    K: negative sample 개수
    반환: 길이 K의 리스트 (각 원소는 단어 index)
    """
    negatives = []
    while len(negatives) < K:
        neg = random.randint(0, V - 1)
        if neg != pos_idx:  # positive context와 같지 않게
            negatives.append(neg)
    return negatives

# -----------------------------
# 3. Word2Vec 모델 정의
# -----------------------------
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)

        # 초기화 (선택사항)
        initrange = 0.5 / embedding_dim
        self.in_embed.weight.data.uniform_(-initrange, initrange)
        self.out_embed.weight.data.uniform_(-0, 0)  # 논문 스타일

    def forward(self, center_idx, context_idx, negative_idx):
        """
        center_idx: (batch,)      예: (1,)
        context_idx: (batch,)     예: (1,)
        negative_idx: (batch, K)  예: (1, K)
        """

        # (batch, embed)
        center = self.in_embed(center_idx)
        context = self.out_embed(context_idx)

        # ------------------
        # 1) Positive loss
        # ------------------
        # 점곱 → (batch,)
        pos_score = torch.sum(center * context, dim=1)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-10)

        # ------------------
        # 2) Negative loss
        # ------------------
        # negative_idx: (batch, K)
        # neg_vectors: (batch, K, embed)
        neg_vectors = self.out_embed(negative_idx)

        # center: (batch, embed) → (batch, embed, 1)
        # bmm 결과: (batch, K, 1) → squeeze(2): (batch, K)
        neg_score = torch.bmm(neg_vectors, center.unsqueeze(2)).squeeze(2)

        # 각 negative에 대해 log σ(-score) 합
        neg_loss = -torch.sum(torch.log(torch.sigmoid(-neg_score) + 1e-10), dim=1)

        # batch 평균
        return (pos_loss + neg_loss).mean()

    def get_embedding(self):
        # input embedding weight 반환 (단어 임베딩)
        return self.in_embed.weight.data

# -----------------------------
# 4. 학습 설정
# -----------------------------
embedding_dim = 20
model = Word2Vec(V, embedding_dim)

optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 100

# -----------------------------
# 5. 학습 루프
# -----------------------------
for epoch in range(epochs):
    total_loss = 0.0

    for center, context in pairs:
        # center, context를 LongTensor (정수형) + batch 차원 추가
        center_t = torch.tensor([center], dtype=torch.long)   # (1,)
        context_t = torch.tensor([context], dtype=torch.long) # (1,)

        # negative samples: 리스트 → (1, K) 텐서
        neg_samples = get_negative_samples(context, 5)   # 길이 5 리스트
        negative_t = torch.tensor(neg_samples, dtype=torch.long).unsqueeze(0)  # (1, 5)

        loss = model(center_t, context_t, negative_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # 20 epoch마다 학습 상황 직관적으로 출력
    if (epoch + 1) % 20 == 0:
        # 학습된 임베딩을 이용해서 positive / negative 점수 비교
        with torch.no_grad():
            in_emb = model.in_embed.weight   # (V, D)
            out_emb = model.out_embed.weight # (V, D)

            pos_scores = []
            neg_scores = []

            for c, ctx in pairs:
                c_vec = in_emb[c]
                ctx_vec = out_emb[ctx]
                pos_score = torch.dot(c_vec, ctx_vec).item()

                # 랜덤 negative 하나 선택
                cand = [i for i in range(V) if i != ctx]
                neg_idx = random.choice(cand)
                neg_vec = out_emb[neg_idx]
                neg_score = torch.dot(c_vec, neg_vec).item()

                pos_scores.append(pos_score)
                neg_scores.append(neg_score)

            avg_pos = sum(pos_scores) / len(pos_scores)
            avg_neg = sum(neg_scores) / len(neg_scores)

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}, "
              f"Avg Pos Score: {avg_pos:.4f}, Avg Neg Score: {avg_neg:.4f}")

# -----------------------------
# 6. 학습된 임베딩 확인
# -----------------------------
embeddings = model.get_embedding()
print("\n[임베딩 weight shape]:", embeddings.shape)  # (V, embedding_dim)

# -----------------------------
# 7. 코사인 유사도 행렬 출력
# -----------------------------
with torch.no_grad():
    # 정규화 (각 단어 벡터를 길이 1로)
    norm_emb = embeddings / embeddings.norm(dim=1, keepdim=True)  # (V, D)
    # 코사인 유사도: (V, V)
    cosine_sim = torch.mm(norm_emb, norm_emb.t())

print("\n[코사인 유사도 행렬] (단어 index 0~9)")
print(cosine_sim)

# -----------------------------
# 8. 각 단어별로 가장 비슷한 단어 Top-3 출력
# -----------------------------
def print_topk_similar(cos_sim_matrix, k=3):
    V = cos_sim_matrix.size(0)
    for i in range(V):
        # 자기 자신(i)는 제외
        sims = cos_sim_matrix[i]
        # 내 index를 제외한 나머지 중에서 top-k
        # torch.topk는 큰 값 순서대로 인덱스를 줌
        values, indices = torch.topk(sims, k + 1)  # 자기 자신 포함해서 k+1개 뽑기
        neighbors = []
        for val, idx in zip(values, indices):
            idx = idx.item()
            if idx == i:
                continue
            neighbors.append((idx, val.item()))
            if len(neighbors) == k:
                break

        print(f"\n단어 {i} 와 가장 비슷한 단어 Top-{k}:")
        for nid, sim in neighbors:
            print(f"  - 단어 {nid} (cosine similarity={sim:.4f})")

print_topk_similar(cosine_sim, k=3)


Epoch 20/100, Loss: 12.5602, Avg Pos Score: 0.1963, Avg Neg Score: -2.8519
Epoch 40/100, Loss: 4.5069, Avg Pos Score: 2.3096, Avg Neg Score: -3.9094
Epoch 60/100, Loss: 1.4501, Avg Pos Score: 3.3304, Avg Neg Score: -5.7074
Epoch 80/100, Loss: 2.4413, Avg Pos Score: 3.7775, Avg Neg Score: -5.0507
Epoch 100/100, Loss: 2.3496, Avg Pos Score: 4.3328, Avg Neg Score: -7.2012

[임베딩 weight shape]: torch.Size([10, 20])

[코사인 유사도 행렬] (단어 index 0~9)
tensor([[ 1.0000,  0.3410,  0.0594,  0.2642,  0.1018,  0.2955,  0.2149,  0.3250,
          0.2633, -0.1319],
        [ 0.3410,  1.0000,  0.3766,  0.3459,  0.3647,  0.3059, -0.0351,  0.3220,
          0.2694,  0.0122],
        [ 0.0594,  0.3766,  1.0000,  0.3550,  0.3747,  0.1894,  0.3584,  0.1820,
          0.3955, -0.2096],
        [ 0.2642,  0.3459,  0.3550,  1.0000,  0.3008,  0.3169,  0.3397,  0.0504,
          0.3543,  0.1319],
        [ 0.1018,  0.3647,  0.3747,  0.3008,  1.0000,  0.2954,  0.1754,  0.3364,
          0.2854,  0.2305],
        [ 0.